# Buck Converter – 400 V → 48 V

This notebook calculates the key parameters of an ideal
buck (step-down) DC-DC converter.

Target specifications:
- Input voltage: 400 V
- Output voltage: 48 V
- Output power: 1 kW

In [1]:
# Parameter definition
Vin = 400.0        # Input voltage [V]
Vout = 48.0        # Output voltage [V]
Pout = 1000.0      # Output power [W]
fsw = 50e3         # Switching frequency [Hz]

ripple_IL = 0.30   # Relative inductor current ripple (30 %)
delta_Vout = 0.5  # Allowed output voltage ripple [V]

print("Parameters set:")
print(f"Vin   = {Vin:.1f} V")
print(f"Vout  = {Vout:.1f} V")
print(f"Pout  = {Pout:.1f} W")
print(f"fsw   = {fsw/1e3:.1f} kHz")


Parameters set:
Vin   = 400.0 V
Vout  = 48.0 V
Pout  = 1000.0 W
fsw   = 50.0 kHz


## Basic Electrical Quantities

From the specified operating point, the average output current,
equivalent load resistance, and duty cycle of the buck converter
are calculated.


In [2]:
# Basic quantities
Iout = Pout / Vout
Rload = Vout / Iout
D = Vout / Vin

print("Basic quantities:")
print(f"Iout  = {Iout:.2f} A")
print(f"Rload = {Rload:.2f} Ohm")
print(f"D     = {D:.3f}")


Basic quantities:
Iout  = 20.83 A
Rload = 2.30 Ohm
D     = 0.120


## Inductor Current Ripple

The buck converter is designed to operate in continuous
conduction mode (CCM).  
The inductor current ripple is chosen as approximately
30 % of the average output current.


In [3]:
# Inductor current ripple
delta_IL = ripple_IL * Iout

print("Inductor current:")
print(f"ΔIL = {delta_IL:.2f} A")

# Inductance calculation
L = (Vin - Vout) * D / (delta_IL * fsw)

print("Inductance:")
print(f"L = {L*1e6:.1f} µH")


Inductor current:
ΔIL = 6.25 A
Inductance:
L = 135.2 µH


## CCM Boundary

To verify the operating mode, the minimum inductance
required for continuous conduction mode (CCM) is calculated.


In [4]:
# CCM boundary inductance
L_ccm = (1 - D) * Rload / (2 * fsw)

print("CCM boundary:")
print(f"L_ccm = {L_ccm*1e6:.1f} µH")

if L > L_ccm:
    print("→ Converter operates safely in CCM")
else:
    print("→ Warning: DCM operation possible")


CCM boundary:
L_ccm = 20.3 µH
→ Converter operates safely in CCM


## Output Capacitor

The output capacitor is sized based on the allowable
output voltage ripple.


In [5]:
# Output capacitor
Cout = delta_IL / (8 * fsw * delta_Vout)

print("Output capacitor:")
print(f"Cout = {Cout*1e6:.1f} µF")


Output capacitor:
Cout = 31.2 µF


## Inductor Current Extremes (CCM)

In continuous conduction mode, the inductor current is fully defined by its average value and peak-to-peak ripple, allowing the maximum and minimum current to be calculated directly from the analytical model.


In [7]:
# --- Inductor current extrema (CCM) ---

# Average inductor (output) current
I_L_avg = Vout / Rload

# Inductor current ripple (peak-to-peak)
Delta_I_L = (Vin - Vout) * D / (L * fsw)

# Maximum and minimum inductor current
I_L_max = I_L_avg + Delta_I_L / 2
I_L_min = I_L_avg - Delta_I_L / 2

print(f"I_L_avg = {I_L_avg:.3f} A")
print(f"Delta_I_L = {Delta_I_L:.3f} A")
print(f"I_L_max = {I_L_max:.3f} A")
print(f"I_L_min = {I_L_min:.3f} A")


I_L_avg = 20.833 A
Delta_I_L = 6.250 A
I_L_max = 23.958 A
I_L_min = 17.708 A


## Summary

The calculated component values represent the analytical
reference for comparison with the LTspice simulation.


In [ ]:
results = {
    "Vin [V]": Vin,
    "Vout [V]": Vout,
    "Pout [W]": Pout,
    "fsw [Hz]": fsw,
    "Duty cycle": D,
    "Iout [A]": Iout,
    "Rload [Ohm]": Rload,
    "ΔIL [A]": delta_IL,
    "L [µH]": L * 1e6,
    "L_ccm [µH]": L_ccm * 1e6,
    "Cout [µF]": Cout * 1e6
}

print("Results:")
for k, v in results.items():
    print(f"{k:14s}: {v:.3f}")


Results:
Vin [V]       : 400.000
Vout [V]      : 48.000
Pout [W]      : 1000.000
fsw [Hz]      : 50000.000
Duty cycle    : 0.120
Iout [A]      : 20.833
Rload [Ohm]   : 2.304
ΔIL [A]       : 6.250
L [µH]        : 135.168
L_ccm [µH]    : 20.275
Cout [µF]     : 31.250


## Advanced Engineering Metrics (Buck Converter)

In addition to the basic steady-state design values (L, C, duty cycle),
practical DC-DC converter design requires further derived quantities.

These parameters are relevant for:
- component selection (MOSFET, diode / synchronous switch, inductor, capacitor)
- thermal and loss estimation
- EMI-aware layout decisions
- validation against time-domain simulation (LTspice)

The following calculations assume:
- continuous conduction mode (CCM)
- idealized triangular inductor current ripple
- no parasitic effects (first-order design stage)

Device-specific parameters such as RDS(on), ESR, DCR or switching energies
must later be replaced by datasheet values.


In [ ]:
# ============================================================
# Advanced derived quantities (losses, RMS currents, limits)
# ============================================================

import math

# Reuse basic quantities from previous cells
Iout = Pout / Vout
Tsw = 1.0 / fsw
Ton = D * Tsw
Toff = (1.0 - D) * Tsw

# ------------------------------------------------------------
# Inductor current characteristics (triangular approximation)
# ------------------------------------------------------------
I_L_avg = Iout
I_L_pk  = I_L_avg + 0.5 * delta_IL
I_L_min = I_L_avg - 0.5 * delta_IL

# RMS current of inductor
# I_rms^2 = I_avg^2 + (ΔI^2)/12 for triangular ripple
I_L_rms = math.sqrt(I_L_avg**2 + (delta_IL**2) / 12.0)

print("Inductor currents:")
print(f"I_L_avg = {I_L_avg:.2f} A")
print(f"I_L_pk  = {I_L_pk:.2f} A")
print(f"I_L_min = {I_L_min:.2f} A")
print(f"I_L_rms = {I_L_rms:.2f} A")

# ------------------------------------------------------------
# RMS currents of switching devices
# ------------------------------------------------------------
# High-side switch conducts during Ton
I_switch_rms = math.sqrt(
    D * (I_L_avg**2 + (delta_IL**2) / 12.0)
)

# Low-side device (diode or synchronous MOSFET) during Toff
I_low_rms = math.sqrt(
    (1.0 - D) * (I_L_avg**2 + (delta_IL**2) / 12.0)
)

print("\nSwitch RMS currents:")
print(f"I_switch_rms (HS) = {I_switch_rms:.2f} A")
print(f"I_low_rms (LS)    = {I_low_rms:.2f} A")

# ------------------------------------------------------------
# Output capacitor RMS ripple current (approximation)
# ------------------------------------------------------------
# RMS value of triangular ripple current component
I_cap_rms = delta_IL / (2.0 * math.sqrt(3.0))

print("\nOutput capacitor ripple:")
print(f"I_cap_rms ≈ {I_cap_rms:.2f} A")

# ------------------------------------------------------------
# Volt-second product of the inductor (core sanity check)
# ------------------------------------------------------------
V_L_on = Vin - Vout
volt_seconds = V_L_on * Ton

print("\nInductor volt-second:")
print(f"V·s_on = {volt_seconds:.3e} V·s")

# ------------------------------------------------------------
# Loss placeholders (replace with datasheet values!)
# ------------------------------------------------------------
Rds_on_HS = 10e-3   # High-side MOSFET Rds(on) [Ohm]
Rds_on_LS = 10e-3   # Low-side MOSFET Rds(on) [Ohm]
ESR_Cout  = 20e-3   # Output capacitor ESR [Ohm]
DCR_L     = 5e-3    # Inductor DC resistance [Ohm]

P_cond_HS = Rds_on_HS * I_switch_rms**2
P_cond_LS = Rds_on_LS * I_low_rms**2
P_cap_ESR = ESR_Cout * I_cap_rms**2
P_L_cu    = DCR_L * I_L_rms**2

print("\nEstimated conduction losses (placeholders):")
print(f"P_cond_HS   = {P_cond_HS:.2f} W")
print(f"P_cond_LS   = {P_cond_LS:.2f} W")
print(f"P_cap_ESR   = {P_cap_ESR:.2f} W")
print(f"P_L_copper  = {P_L_cu:.2f} W")

# ------------------------------------------------------------
# Control and EMI-relevant guideline values
# ------------------------------------------------------------
f_c_guideline = fsw / 10.0

print("\nControl & EMI guidelines:")
print(f"Suggested control bandwidth ≈ {f_c_guideline:.0f} Hz")
print("Primary loop area: keep as small as possible")
print("High di/dt loop: MOSFET – diode – input capacitor")


Inductor currents:
I_L_avg = 20.83 A
I_L_pk  = 23.96 A
I_L_min = 17.71 A
I_L_rms = 20.91 A

Switch RMS currents:
I_switch_rms (HS) = 7.24 A
I_low_rms (LS)    = 19.62 A

Output capacitor ripple:
I_cap_rms ≈ 1.80 A

Inductor volt-second:
V·s_on = 8.448e-04 V·s

Estimated conduction losses (placeholders):
P_cond_HS   = 0.52 W
P_cond_LS   = 3.85 W
P_cap_ESR   = 0.07 W
P_L_copper  = 2.19 W

Control & EMI guidelines:
Suggested control bandwidth ≈ 5000 Hz
Primary loop area: keep as small as possible
High di/dt loop: MOSFET – diode – input capacitor
